In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.signal import savgol_filter

# 1. Load file
file_path = 'RhoA_actin_dynamics.xlsx'
df = pd.read_excel(file_path) if file_path.endswith('.xlsx') else pd.read_csv(file_path)

dt = df['Time'].diff().mean()
time_arr = df['Time'].values

# Automatic detection of ROI pairs
roi_pairs = [(c.replace('_Actin', ''), c, f"{c.replace('_Actin', '')}_RhoA")
             for c in df.columns if '_Actin' in c and f"{c.replace('_Actin', '')}_RhoA" in df.columns]

# --- Analysis parameters ---
SMOOTH_WINDOW = 9   # Differentiation amplifies noise, so slightly stronger smoothing (around 9-11) is recommended
SEARCH_WINDOW = 90  # Rate analysis tends to yield larger lags, so widen the search window to ±90 s

corr_list_rate = []
results_summary_rate = []

for roi_name, actin_col, rhoa_col in roi_pairs:
    actin_raw = df[actin_col].values
    rhoa_raw = df[rhoa_col].values

    # 1. Smoothing
    actin_sm = savgol_filter(actin_raw, window_length=SMOOTH_WINDOW, polyorder=2)
    rhoa_sm = savgol_filter(rhoa_raw, window_length=SMOOTH_WINDOW, polyorder=2)

    # 2. Calculate rate of change (derivative)
    # dActin/dt: Actin accumulation rate
    d_actin = np.gradient(actin_sm, dt)
    # -dRhoA/dt: RhoA decay rate (negative sign applied to treat decrease as positive values)
    d_rhoa_decay = -np.gradient(rhoa_sm, dt)

    # 3. Z-score normalization
    act_rate_norm = (d_actin - np.mean(d_actin)) / np.std(d_actin)
    rho_rate_norm = (d_rhoa_decay - np.mean(d_rhoa_decay)) / np.std(d_rhoa_decay)

    # 4. Rate-based Cross-correlation
    corr = signal.correlate(rho_rate_norm, act_rate_norm, mode='full') / len(act_rate_norm)
    lags = signal.correlation_lags(len(act_rate_norm), len(rho_rate_norm), mode='full')
    time_lags = lags * dt

    # Peak search
    valid_mask = (time_lags >= -SEARCH_WINDOW) & (time_lags <= SEARCH_WINDOW)
    valid_lags = time_lags[valid_mask]
    valid_corr = corr[valid_mask]

    max_idx = np.argmax(valid_corr)
    peak_lag = valid_lags[max_idx]
    max_corr = valid_corr[max_idx]

    corr_list_rate.append(corr)
    results_summary_rate.append({
        'ROI': roi_name,
        'Rate Phase Lag (s)': peak_lag,
        'Max Correlation (r)': max_corr
    })

# Combine into summary DataFrame
summary_df_rate = pd.DataFrame(results_summary_rate)

# Mean waveform across all ROIs
corr_matrix_rate = np.array(corr_list_rate)
mean_corr_rate = np.mean(corr_matrix_rate, axis=0)

valid_mask_mean = (time_lags >= -SEARCH_WINDOW) & (time_lags <= SEARCH_WINDOW)
valid_lags_mean = time_lags[valid_mask_mean]
valid_mean_corr_rate = mean_corr_rate[valid_mask_mean]

overall_max_idx = np.argmax(valid_mean_corr_rate)
overall_peak_lag = valid_lags_mean[overall_max_idx]
overall_max_corr = valid_mean_corr_rate[overall_max_idx]

print("=== Rate-based Cross-Correlation Analysis Results ===")
print(summary_df_rate.to_string(index=False))

mean_lag = summary_df_rate['Rate Phase Lag (s)'].mean()
sem_lag = summary_df_rate['Rate Phase Lag (s)'].std() / np.sqrt(len(summary_df_rate))
mean_r = summary_df_rate['Max Correlation (r)'].mean()

print("\n--- Statistical Results ---")
print(f"All ROIs Mean Rate Phase Lag: {mean_lag:.2f} ± {sem_lag:.2f} s")
print(f"All ROIs Mean Max Correlation (r): {mean_r:.3f}")
print(f"Overall Peak Lag from mean curve: {overall_peak_lag:.2f} s (Max r = {overall_max_corr:.3f})")


# --- Plotting graphs ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), dpi=300)

# [Spine and tick line width settings]
FRAME_WIDTH = 1.5  # Frame linewidth

for ax in axes:
    # Make plot frame spines (top, bottom, left, right) thicker
    for spine in ax.spines.values():
        spine.set_linewidth(FRAME_WIDTH)
    # Make axis tick lines thicker to match the frame
    ax.tick_params(width=FRAME_WIDTH)

# (Panel A) Rate cross-correlation curves for individual ROIs and mean
for i, corr in enumerate(corr_list_rate):
    axes[0].plot(time_lags, corr, color='gray', alpha=0.35, linewidth=1,
                 label='Individual ROI' if i == 0 else "")
axes[0].plot(time_lags, mean_corr_rate, color='darkorange', linewidth=2.5, label='Mean Rate Cross-Correlation')
axes[0].axvline(x=overall_peak_lag, color='red', linestyle='--', label=f'Peak Lag = {overall_peak_lag:.1f} s')
axes[0].axhline(y=0, color='black', linestyle=':', alpha=0.5)
axes[0].set_xlim(-SEARCH_WINDOW, SEARCH_WINDOW)
axes[0].set_xlabel('Time Lag (s)\n(Positive = RhoA decay rate lags behind Actin recovery rate)')
axes[0].set_ylabel('Cross-Correlation Coefficient (r)')
axes[0].set_title('A. Rate-based Cross-Correlation (dActin/dt vs -dRhoA/dt)')

# --- Position legend outside ---
axes[0].legend(fontsize=8, bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)

# Disable grid lines
axes[0].grid(False)

# (Panel B) Distribution of Rate Phase Lag (boxplot)
axes[1].boxplot(summary_df_rate['Rate Phase Lag (s)'], positions=[1], patch_artist=True,
                boxprops=dict(facecolor='navajowhite', alpha=0.7, linewidth=FRAME_WIDTH),
                medianprops=dict(linewidth=FRAME_WIDTH),
                whiskerprops=dict(linewidth=FRAME_WIDTH),
                capprops=dict(linewidth=FRAME_WIDTH))
axes[1].scatter(np.ones(len(summary_df_rate)), summary_df_rate['Rate Phase Lag (s)'], color='darkred', zorder=3)
axes[1].axhline(y=0, color='gray', linestyle=':')
axes[1].set_xticks([1])
axes[1].set_xticklabels(['ROIs'])
axes[1].set_ylabel('Rate Phase Lag (s)')
axes[1].set_title(f'B. Rate Phase Lag Distribution\n(Mean: {mean_lag:.1f} ± {sem_lag:.1f} s)')

# Disable grid lines
axes[1].grid(False)

# Adjust layout so legends and axis labels do not clip
plt.tight_layout()
plt.show()